# Simple Networks --- Instructor Solutions

This is the **instructor answer key** for the three exercises in
[`notebooks/02_simple_networks.ipynb`](../notebooks/02_simple_networks.ipynb) (MMSB §2.1.3,
"Simple Networks"). For each exercise it gives:

1. a full hand derivation (not just the answer --- the *reasoning*, since that's what you'd
   want to check for or reconstruct on a whiteboard during office hours),
2. the completed code cell, runnable end-to-end, and
3. **teaching notes**: what a fully-correct submission should show, the most common
   student mistakes, and a couple of talking points/extensions if you have time in section.

**Not for distribution to students before the exercise deadline.** This notebook is
deliberately excluded from the published Quarto site (see `_quarto.yml`'s
`project.render` list) so it doesn't show up on GitHub Pages alongside the student
notebook, but it is still committed to the repo history like any other file.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp


## Exercise 1: Three-Step Chain $A \to B \to C$

### Derivation

The only genuinely new part relative to Example III is that $B$ now has *two* terms
instead of one: it gains what $A$ loses to it, and loses what it hands on to $C$.
Writing "rate of change = rate in $-$ rate out" for each species:

$$
\frac{dA}{dt} = -k_1 A \qquad\text{($A$ only loses, to $B$)}
$$
$$
\frac{dB}{dt} = \underbrace{k_1 A}_{\text{in, from }A} - \underbrace{k_2 B}_{\text{out, to }C}
$$
$$
\frac{dC}{dt} = k_2 B \qquad\text{($C$ only gains, from $B$)}
$$

Nothing enters or leaves the $A$-$B$-$C$ system from outside, so the total is conserved
for all $t$, not just asymptotically:

$$
A(t) + B(t) + C(t) = A_0 + B_0 + C_0.
$$

This is exactly the conservation check used in Example III, just extended to three
species instead of two.


In [ ]:
# --- Parameters (edit and re-run) ---
k1 = 0.5      # A -> B rate constant (1/time)
k2 = 0.2      # B -> C rate constant (1/time)
A0, B0, C0 = 1.0, 0.0, 0.0
t_span = (0, 20)
t_eval = np.linspace(*t_span, 300)

def chain_rhs(t, y, k1, k2):
    A, B, C = y
    dA = -k1 * A
    dB = k1 * A - k2 * B   # gains from A, loses to C
    dC = k2 * B            # only gains, from B
    return [dA, dB, dC]

sol = solve_ivp(chain_rhs, t_span, [A0, B0, C0], t_eval=t_eval, args=(k1, k2))
A_numeric, B_numeric, C_numeric = sol.y

total0 = A0 + B0 + C0
total_final = A_numeric[-1] + B_numeric[-1] + C_numeric[-1]
print(f"Conservation A+B+C: start = {total0:.3f}, end = {total_final:.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(t_eval, A_numeric, label="A", lw=2)
ax.plot(t_eval, B_numeric, label="B", lw=2)
ax.plot(t_eval, C_numeric, label="C", lw=2)
ax.set_xlabel("time $t$")
ax.set_ylabel("concentration")
ax.set_title(r"Exercise 1 (solved): Chain $A \to B \to C$")
ax.legend()
ax.grid(alpha=0.3)
plt.show()


### Teaching notes

**What a correct submission looks like:** the conservation print line matches to
several decimal places (small drift, if any, is integrator tolerance, not a bug), and
the plot shows $A$ decaying monotonically, $B$ rising then falling (a transient "hump"
--- it is produced by $A$ faster than it is consumed by $C$ at first, then the reverse),
and $C$ rising monotonically to $A_0+B_0+C_0$.

**Common mistakes:**

- Writing `dB = k1 * A - k2 * A` (reusing `A` instead of `B` in the loss term) --- this
  usually still "runs" without the `NotImplementedError`, so it's easy to miss unless
  you check the conservation print line, which will visibly fail to hold.
- Dropping the minus sign on the consumption term in `dB`, i.e. `dB = k1*A + k2*B` ---
  the plot still looks plausible-ish (line still rises) unless a student happens to
  plot it long enough to notice $B$ never turns over.
- Leaving `total0`/`total_final` as plain numbers copied from a print output rather
  than expressions in the variables --- works for this one run, but silently breaks if
  the parameters are changed and re-run, defeating the point of the sanity check.

**Extension, if there's time:** ask what happens to the height and timing of $B$'s
peak as $k_2 \to k_1$ vs. $k_2 \gg k_1$ or $k_2 \ll k_1$ --- this previews the
"rate-limiting step" intuition used later for enzyme cascades.


## Exercise 2: Production, Decay, and Reversible Conversion

### Derivation

$$
\frac{dA}{dt} = k_p - k_dA - k_fA + k_rB, \qquad \frac{dB}{dt} = k_fA - k_rB.
$$

**The "obvious" route** is to set both derivatives to zero and solve the 2x2 linear
system directly. From $dB/dt=0$:

$$
B_{ss} = \frac{k_f}{k_r} A_{ss}.
$$

Substituting into $dA/dt = 0$:

$$
k_p - k_dA_{ss} - k_fA_{ss} + k_r\left(\frac{k_f}{k_r}A_{ss}\right) = 0
\;\Longrightarrow\;
k_p - k_dA_{ss} - k_fA_{ss} + k_fA_{ss} = 0
\;\Longrightarrow\;
k_p = k_dA_{ss}.
$$

The $\pm k_fA_{ss}$ terms cancel exactly, leaving

$$
\boxed{A_{ss} = \frac{k_p}{k_d}}, \qquad
\boxed{B_{ss} = \frac{k_f}{k_r}\cdot\frac{k_p}{k_d} = \frac{k_pk_f}{k_dk_r}}.
$$

**A faster route, worth showing in section:** add the two ODEs together. The $\pm k_fA$
and $\pm k_rB$ terms cancel *before* you ever set anything to zero:

$$
\frac{d(A+B)}{dt} = k_p - k_dA.
$$

At steady state $d(A+B)/dt = 0$ forces $A_{ss}=k_p/k_d$ **immediately**, with no need to
solve a 2x2 system at all --- the conversion terms are internal to the $A\leftrightarrow B$
pool and can never appear in the equation for the pool's total, since they only move
mass between $A$ and $B$, never in or out of the pool. Only $k_p$ (production into the
pool) and $k_d$ (decay out of the pool, and only through $A$) can set the total. This
is the same "conserved-quantity" reasoning as Examples III/IV, just applied to a pool
that itself has external production/decay.

**Sanity check against the hint:** setting $k_f=k_r=0$ gives $A_{ss}=k_p/k_d$ and
(taking the $0/0$ in $B_{ss}$ as its limiting sense, no conversion pathway exists at all)
recovers Example II exactly, as the exercise hint promises.

### A grading gotcha: how long is "long enough" to reach steady state?

Unlike Example II, where the approach to steady state has the single, simple time
constant $1/k_d$, here $A$ and $B$ relax with the *two* eigenvalues of

$$
M = \begin{pmatrix}-(k_d+k_f) & k_r \\ k_f & -k_r\end{pmatrix},
$$

and it is the **slower** of the two that sets how long you must integrate before the
numeric values are safe to compare against $A_{ss}, B_{ss}$. A short calculation gives
a clean closed form for $\det M$:

$$
\det M = (k_d+k_f)k_r - k_fk_r = k_dk_r,
\qquad
\operatorname{tr}M = -(k_d+k_f+k_r),
$$

so the eigenvalues are $\lambda_{\pm} = \tfrac12\!\left(\operatorname{tr}M \pm
\sqrt{\operatorname{tr}M^2-4\det M}\right)$, and the slow relaxation rate is
$|\lambda_+|$ (the one closer to zero). For the parameters below,
$|\lambda_+|\approx 0.1$, i.e. a time constant of $\sim\!10$ --- **more than three
times** the naive $1/k_d\approx 3.3$ a student might assume by analogy with Example
II. `t_span=(0, 20)` is only about two of these slow time constants and visibly
undershoots both steady states; `t_span=(0, 100)` (ten time constants) is needed for
the numeric values to actually match the theory below to 3-4 decimals. **If a
student's printed numeric/theory comparison is systematically off by the same
direction and rough magnitude for both $A$ and $B$ (numeric below theory when both
are positive), suspect insufficient integration time before suspecting their algebra.**


In [ ]:
# --- Parameters (edit and re-run) ---
kp = 1.0      # production rate of A (amount/time)
kd = 0.3      # decay rate constant of A (1/time)
kf = 0.6      # forward conversion A -> B (1/time)
kr = 0.4      # reverse conversion B -> A (1/time)
A0, B0 = 0.0, 0.0
t_span = (0, 100)  # needs to be long relative to the slower of the system's two relaxation rates
t_eval = np.linspace(*t_span, 300)

def prod_decay_reversible_rhs(t, y, kp, kd, kf, kr):
    A, B = y
    dA = kp - kd * A - kf * A + kr * B
    dB = kf * A - kr * B
    return [dA, dB]

sol = solve_ivp(prod_decay_reversible_rhs, t_span, [A0, B0], t_eval=t_eval,
                 args=(kp, kd, kf, kr))
A_numeric, B_numeric = sol.y

A_ss_theory = kp / kd
B_ss_theory = (kf / kr) * A_ss_theory
print(f"A_ss: theory = {A_ss_theory:.4f}, numeric (final value) = {A_numeric[-1]:.4f}")
print(f"B_ss: theory = {B_ss_theory:.4f}, numeric (final value) = {B_numeric[-1]:.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(t_eval, A_numeric, label="A", lw=2)
ax.plot(t_eval, B_numeric, label="B", lw=2)
ax.axhline(A_ss_theory, color="C0", ls=":", alpha=0.7)
ax.axhline(B_ss_theory, color="C1", ls=":", alpha=0.7)
ax.set_xlabel("time $t$")
ax.set_ylabel("concentration")
ax.set_title(r"Exercise 2 (solved): Production/Decay + Reversible Conversion")
ax.legend()
ax.grid(alpha=0.3)
plt.show()


### Teaching notes

**What a correct submission looks like:** both printed theory/numeric pairs agree to
~3-4 decimal places, and the plot shows both curves flattening onto the dotted
steady-state lines. Note $A_{ss}$ does **not** depend on $k_f$ or $k_r$ at all --- a
student who gets a $k_f$ or $k_r$ into their $A_{ss}$ formula has made an algebra slip,
most often forgetting the $\pm k_fA$ cancellation above.

**Common mistakes:**

- Solving $dA/dt=0$ for $A_{ss}$ while treating $B$ as a free constant instead of
  substituting the *other* equation's relationship first --- leads to one equation in
  two unknowns and a formula that "looks" right but still has a stray $B$ or $B_{ss}$
  in it.
- Sign error on the reverse term, writing $dA/dt = k_p - k_dA - k_fA - k_rB$ (minus
  instead of plus) --- this breaks conservation of the "obvious route" cancellation
  above and gives a visibly wrong $A_{ss}$ that depends on $k_f, k_r$.
- Reading off "steady state" from the plot at $t=20$ without checking the curve has
  actually flattened --- if a student's $k_d$ is small, $t=20$ may not be long enough;
  worth asking them to extend `t_span` and confirm the numeric value stops moving.

**Extension, if there's time:** ask students to derive $A_{ss}$ by the "add the
equations" shortcut *themselves*, after they've already done it the slow way --- it's a
good moment to introduce the general idea that a linear combination of state variables
can obey a much simpler ODE than any individual state variable does (a preview of
using conserved quantities / linear invariants to reduce model dimension, which comes
back in later chapters).


## Exercise 3: Competing Decay Pathways

### Derivation

$A$ loses mass down *two* independent first-order pathways at once, so its total loss
rate is the sum of both:

$$
\frac{dA}{dt} = -k_1A - k_2A = -(k_1+k_2)A, \qquad
\frac{dP}{dt} = k_1A, \qquad
\frac{dQ}{dt} = k_2A.
$$

**(a) How fast does $A$ disappear overall?** The $A$ equation is exactly Example I's
pure decay equation with an *effective* rate constant $k_1+k_2$:

$$
A(t) = A_0e^{-(k_1+k_2)t}, \qquad t_{1/2} = \frac{\ln 2}{k_1+k_2}.
$$

Note this is **faster** than either pathway alone would give ($t_{1/2}$ shrinks as
either $k_1$ or $k_2$ grows) --- competing pathways always speed up the overall decay
of the shared reactant, even though each individual pathway's rate constant is
unchanged.

**(b) The ratio $Q(t)/P(t)$.** Rather than solving for $P(t)$ and $Q(t)$ separately and
then dividing, divide the ODEs directly:

$$
\frac{dQ/dt}{dP/dt} = \frac{k_2A}{k_1A} = \frac{k_2}{k_1}
\quad\text{(constant, for all $t$ with $A(t)\neq 0$).}
$$

This is a **stronger** statement than the exercise's hint asks for: since $P(0)=Q(0)=0$,
integrating $dQ/dP = k_2/k_1$ gives $Q(t) = (k_2/k_1)P(t)$ for *every* $t>0$, not only
in the long-time limit. (If a sharp student notices and asks about this, it's worth
confirming --- the ratio is exactly constant here because $P$ and $Q$ are driven by the
*same* $A(t)$ with no other source or sink; that would stop being true if, say, $Q$ had
its own additional production term.)


In [ ]:
# --- Parameters (edit and re-run) ---
k1 = 0.4      # A -> P rate constant (1/time)
k2 = 0.1      # A -> Q rate constant (1/time)
A0 = 1.0
P0, Q0 = 0.0, 0.0
t_span = (0, 20)
t_eval = np.linspace(*t_span, 300)

def competing_rhs(t, y, k1, k2):
    A, P, Q = y
    dA = -(k1 + k2) * A
    dP = k1 * A
    dQ = k2 * A
    return [dA, dP, dQ]

sol = solve_ivp(competing_rhs, t_span, [A0, P0, Q0], t_eval=t_eval, args=(k1, k2))
A_numeric, P_numeric, Q_numeric = sol.y

# Check (a): effective half-life
t_half_theory = np.log(2) / (k1 + k2)
t_half_numeric = np.interp(A0 / 2, A_numeric[::-1], t_eval[::-1])
print(f"Half-life: theory = {t_half_theory:.3f}, numeric = {t_half_numeric:.3f}")

# Check (b): ratio Q/P, evaluated away from t=0 to avoid 0/0
mask = P_numeric > 1e-9
ratio_theory = k2 / k1
ratio_numeric = Q_numeric[mask] / P_numeric[mask]
print(f"Q/P ratio: theory = {ratio_theory:.4f}, "
      f"numeric range = [{ratio_numeric.min():.4f}, {ratio_numeric.max():.4f}]")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(t_eval, A_numeric, label="A", lw=2)
ax.plot(t_eval, P_numeric, label="P", lw=2)
ax.plot(t_eval, Q_numeric, label="Q", lw=2)
ax.set_xlabel("time $t$")
ax.set_ylabel("concentration")
ax.set_title(r"Exercise 3 (solved): Competing Decay Pathways")
ax.legend()
ax.grid(alpha=0.3)
plt.show()


### Teaching notes

**What a correct submission looks like:** the half-life check matches to ~3 decimal
places, and the $Q/P$ ratio is essentially constant across the whole evaluated range
(tight around `k2/k1`, not just "close by the end") --- this constancy for *all* $t>0$,
not merely as $t\to\infty$, is the detail most solutions will miss since the exercise
prompt itself only asks about the "long-time ratio."

**Common mistakes:**

- Writing `dA = -k1*A - k2*A` (correct) but then `dP = k1*A - k2*A` or similar
  cross-talk between the two product equations --- a giveaway is a $P$ or $Q$ curve
  that doesn't monotonically approach a plateau at $A_0 \cdot k_i/(k_1+k_2)$.
- Computing the $Q/P$ ratio starting at $t=0$ without guarding against $P(0)=0$,
  producing a `nan` or `inf` in the first entry and a spurious "it's not constant"
  conclusion --- worth pointing out `mask = P_numeric > 1e-9`-style filtering as a
  standard pattern whenever dividing two quantities that both start at zero.
- Treating this as two independent decay problems for $A$ (one governed by $k_1$, one
  by $k_2$) instead of one $A$ pool with a single combined effective rate constant.

**Extension, if there's time:** since $P_\infty = A_0\cdot k_1/(k_1+k_2)$ and
$Q_\infty = A_0\cdot k_2/(k_1+k_2)$ (branching-ratio partition of the initial pool),
ask students to check this against their simulation's final values too --- a second,
independent numeric confirmation of the same $k_2/k_1$ result, from the endpoint
values this time rather than the whole-trajectory ratio.


## Grading rubric summary

| Exercise | Full marks requires | Partial credit for |
|---|---|---|
| 1. Chain $A\to B\to C$ | Correct `dB`, `dC`; conservation check written in terms of variables (not hardcoded numbers); plot shows $B$'s transient peak | Correct ODEs but conservation check hardcodes a number from one run |
| 2. Production/decay + reversible | Correct RHS function; $A_{ss}, B_{ss}$ derived and matching numeric to ~3 decimals; plot with steady-state reference lines | Correct simulation but $A_{ss}$/$B_{ss}$ derivation has an algebra slip (often a stray $k_f$/$k_r$ in $A_{ss}$) |
| 3. Competing decay | Correct three-ODE RHS; both checks (half-life *and* ratio) implemented; ratio checked away from $t=0$ | Correct simulation and half-life check, but ratio check computed only at $t=t_{\text{final}}$ instead of showing it holds throughout |

**Summary table for §2.1.3 as a whole**, extending the one in the student notebook:

| Exercise | ODE(s) | Key result |
|---|---|---|
| 1. Chain $A\to B\to C$ | $\dot A=-k_1A,\ \dot B=k_1A-k_2B,\ \dot C=k_2B$ | $A+B+C$ conserved; $B$ shows a transient peak |
| 2. Production/decay + reversible | $\dot A=k_p-k_dA-k_fA+k_rB,\ \dot B=k_fA-k_rB$ | $A_{ss}=k_p/k_d$ (independent of $k_f,k_r$); $B_{ss}=k_pk_f/(k_dk_r)$ |
| 3. Competing decay | $\dot A=-(k_1+k_2)A,\ \dot P=k_1A,\ \dot Q=k_2A$ | Effective half-life $\ln2/(k_1+k_2)$; $Q/P=k_2/k_1$ for all $t>0$ |

**Reference:** Ingalls, B. P. (2013). *Mathematical Modeling in Systems Biology: An
Introduction*. MIT Press. Chapter 2, §2.1.3. Official PDF (with solutions):
<https://www.math.uwaterloo.ca/~bingalls/MMSB/MMSB_w_solutions.pdf>
